In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout, LSTM



In [ ]:
DATA_PATH = "data/cleaned/binary_cleaned.csv"

df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
X = df["text"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

X_test, X_val, y_test, y_val = train_test_split(
X_test, y_test, test_size=0.5, random_state=42, stratify=y_test
)

print("Train:", len(X_train))
print("Val :", len(X_val))
print("Test :", len(X_test))

In [ ]:
MAX_VOCAB = 20000
MAX_LEN = 100  

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

train_seq = tokenizer.texts_to_sequences(X_train)
test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_test_pad  = pad_sequences(test_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_val_pad = pad_sequences(tokenizer.texts_to_sequences(X_val), maxlen=MAX_LEN, padding="post", truncating="post")

y_train = np.array(y_train)
y_test  = np.array(y_test)
y_val = np.array(y_val)

print(X_train_pad.shape, X_val_pad.shape, X_test_pad.shape)

In [ ]:
GLOVE_PATH = "embeddings/glove.6B.200d.txt"
EMB_DIM = 200

embeddings_index = {}
with open(GLOVE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype="float32")
        embeddings_index[word] = vector

word_index = tokenizer.word_index
num_words = min(MAX_VOCAB, len(word_index) + 1)

embedding_matrix = np.zeros((num_words, EMB_DIM))
for word, i in word_index.items():
    if i >= MAX_VOCAB:
        continue
    vec = embeddings_index.get(word)
    if vec is not None:
        embedding_matrix[i] = vec

print("Embedding matrix:", embedding_matrix.shape)

In [ ]:
model = Sequential([
    Embedding(
        input_dim=num_words,
        output_dim=EMB_DIM,
        weights=[embedding_matrix],
        input_length=MAX_LEN,
        trainable=False
    ),

    LSTM(128),
    Dropout(0.5),

    Dense(64, activation="relu"),
    Dropout(0.3),

    Dense(1, activation="sigmoid")
])

model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
model.summary()


In [ ]:
history = model.fit(
    X_train_pad, y_train,
    epochs=8,
    batch_size=64,
    validation_data=(X_val_pad, y_val),
    verbose=1
)

In [ ]:
loss, acc = model.evaluate(X_test_pad, y_test, verbose=0)
print("Test Loss:", loss)
print("Test Accuracy:", acc)

In [ ]:
y_pred_prob = model.predict(X_test_pad)
y_pred = (y_pred_prob > 0.5).astype(int).reshape(-1)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
# Accuracy curve
plt.figure(figsize=(8,5))
plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Learning Curve - Accuracy")
plt.legend()
plt.grid(True)
plt.show()

# Loss curve
plt.figure(figsize=(8,5))
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Learning Curve - Loss")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import pickle

with open("artifacts/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [ ]:
np.save("artifacts/X_train.npy", X_train_pad)
np.save("artifacts/X_val.npy",   X_val_pad)
np.save("X_test.npy",  X_test_pad)

np.save("artifacts/y_train.npy", y_train)
np.save("artifacts/y_val.npy",   y_val)
np.save("artifacts/y_test.npy",  y_test)

print("Saved all .npy files ✅")

In [ ]:
model.save("artifacts/sentiment_model.keras")
print("Saved sentiment_model.keras ✅")